In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round
import re

spark = SparkSession.builder \
    .appName("Amazon Product Review Analytics") \
    .getOrCreate()

file_path = "AmazonReviews.txt"


raw_text = spark.sparkContext.wholeTextFiles(file_path).map(lambda x: x[1])

def parse_reviews(text):

    records = text.split("\n\n")
    parsed_records = []

    for record in records:
        if not record.strip():
            continue

        prod_id, title, score = None, None, None

        for line in record.strip().split("\n"):
            if line.startswith("product/productId:"):
                prod_id = line.replace("product/productId:", "").strip()
            elif line.startswith("product/title:"):
                title = line.replace("product/title:", "").strip()
            elif line.startswith("review/score:"):
                try:
                    score = float(line.replace("review/score:", "").strip())
                except ValueError:
                    score = None

        if prod_id and score is not None:
            title = title if title else "Unknown Title"
            parsed_records.append((prod_id, title, score))

    return parsed_records

formatted_rdd = raw_text.flatMap(parse_reviews)

df = spark.createDataFrame(
    formatted_rdd,
    ["product_id", "title", "score"]
)

print("\nRaw Data (Sécurisé):")
df.show(10, truncate=False)

result = df.groupBy(
    "product_id",
    "title"
).agg(
    round(avg("score"), 2).alias("avg_review_score"),
    count("score").alias("total_reviews")
)

print("\nAverage review score per product:")
result.orderBy(
    result.total_reviews.desc()
).show(20, truncate=False)

spark.stop()